In [1]:
import argparse
import os
import sys
import time

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from sklearn.cluster import KMeans
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score

# ------------------------- CONFIG -------------------------
NK = 23
CLIM_VARS = ["chirps_z", "dry_z", "sm1_z", "sm2_z", "t2m_z", "vpd_z", "lst_z", "spi32", "spi64", "spi96"]
N_CLIM_LAGS = 3            
N_NDVI_LAGS = 2           
S1_VARS = ["vv_z", "vh_z", "cr_z", "rvi_z"]
N_S1_LAGS = 3              
N_YEAR_GROUPS = 5
BUFFER_DEG = 0.15          
SEED = 42


# ------------------------- data -------------------------
def read_table(d, stem):
    pq, csv = os.path.join(d, stem + ".parquet"), os.path.join(d, stem + ".csv")
    df = pd.read_parquet(pq) if os.path.exists(pq) else pd.read_csv(csv, dtype={"pid": str})
    df["pid"] = df["pid"].astype(str)
    return df


def prepare(feat_dir, label_dir, feat_file="features_long", min_year=None, target="stress"):
    feat = read_table(feat_dir, feat_file)
    lab = read_table(label_dir, "label_long")
    feat["t"] = feat["year"] * NK + feat["k"]
    lab["t"] = lab["year"] * NK + lab["k"]
    feat = feat.drop_duplicates(["pid", "t"])
    lab = lab[lab["pid"].isin(feat["pid"].unique())].dropna(subset=["z"])
    if target == "sustained":
        # stress that lasts: this window AND the next one are below z = -1 (filters one-window satellite noise).
        # Rows with no next window (the last label window of a season) are dropped.
        z_next = lag_values(lab.set_index(["pid", "t"]), "z", lab["pid"].to_numpy(), lab["t"].to_numpy(), -1)
        lab = lab.assign(z_next=z_next)
        lab = lab[lab["z_next"].notna()]
        print("Target = SUSTAINED stress (z < -1 now and in the next 16-day window)")
    if min_year:
        lab = lab[lab["year"] >= min_year]
        print(f"Rows restricted to years >= {min_year} (all feature sets and baselines use the same rows)")
    lab = lab.reset_index(drop=True)
    lab["y"] = (lab["z"] < -1.0).astype(int)
    if target == "sustained":
        lab["y"] = ((lab["z"] < -1.0) & (lab["z_next"] < -1.0)).astype(int)

    # spatial blocks from CHIRPS cells
    cells = feat.groupby("cell")[["lon", "lat"]].mean()
    km = KMeans(n_clusters=N_BLOCKS, n_init=10, random_state=SEED).fit(cells[["lon", "lat"]])
    cells["block"] = km.labels_
    pid_cell = feat.drop_duplicates("pid").set_index("pid")["cell"]
    lab["cell"] = lab["pid"].map(pid_cell)
    lab = lab[lab["cell"].notna()].copy()
    lab["cell"] = lab["cell"].astype(int)
    lab["block"] = lab["cell"].map(cells["block"]).astype(int)
    print(f"Rows: {len(lab):,}  points: {lab['pid'].nunique():,}  cells: {len(cells)}  "
          f"base stress rate: {lab['y'].mean():.3f}")
    return feat, lab, cells


def lag_values(indexed, col, pids, t, L):
    idx = pd.MultiIndex.from_arrays([pids, np.asarray(t) - L])
    return indexed[col].reindex(idx).to_numpy(np.float32)


def build_xy(feat, lab, H):
    """Return dict of feature blocks for horizon H (windows)."""
    F = feat.set_index(["pid", "t"])
    Z = lab.set_index(["pid", "t"])
    pids, t = lab["pid"].to_numpy(), lab["t"].to_numpy()

    clim = {}
    for L in range(H, H + N_CLIM_LAGS):
        for c in CLIM_VARS:
            clim[f"{c}_L{L}"] = lag_values(F, c, pids, t, L)
    clim["k"] = lab["k"].to_numpy(np.float32)
    ndvi = {}
    for L in range(H + 1, H + 1 + N_NDVI_LAGS):
        ndvi[f"ndvi_z_L{L}"] = lag_values(Z, "z", pids, t, L)

    return pd.DataFrame(clim), pd.DataFrame(ndvi)


def build_s1(feat, lab, H):
    """Sentinel-1 anomaly block (needs features_long_s1 from 03b)."""
    missing = [c for c in S1_VARS if c not in feat.columns]
    if missing:
        raise ValueError(f"S1 columns {missing} not in the feature table; run 03b and pass --feat-file features_long_s1")
    F = feat.set_index(["pid", "t"])
    pids, t = lab["pid"].to_numpy(), lab["t"].to_numpy()
    out = {}
    for L in range(H, H + N_S1_LAGS):
        for c in S1_VARS:
            out[f"{c}_L{L}"] = lag_values(F, c, pids, t, L)
    return pd.DataFrame(out)


# ------------------------- CV -------------------------
def make_folds(lab, cells, scheme, buffer_deg=BUFFER_DEG, n_year_groups=N_YEAR_GROUPS, eval_min_year=None):
    years = np.sort(lab["year"].unique())
    eval_years = years[years >= eval_min_year] if eval_min_year else years
    yr, blk, cell = lab["year"].to_numpy(), lab["block"].to_numpy(), lab["cell"].to_numpy()
    coords = cells[["lon", "lat"]]
    cell_ids = cells.index.to_numpy()
    cell_xy = coords.to_numpy()
    cell_block = cells["block"].to_numpy()

    def buffered_exclude(b):
        test_xy = cell_xy[cell_block == b]
        d, _ = cKDTree(test_xy).query(cell_xy)
        near = set(cell_ids[(d <= buffer_deg)])
        return np.isin(cell, list(near))

    folds = []
    if scheme == "loso":
        for y in eval_years:
            folds.append((f"year{y}", yr != y, yr == y))
    elif scheme == "spatial":
        for b in range(N_BLOCKS):
            test = (blk == b) & ((yr >= eval_min_year) if eval_min_year else True)
            folds.append((f"block{b}", ~buffered_exclude(b), test))
    elif scheme == "st":
        groups = np.array_split(eval_years, n_year_groups)
        for gi, g in enumerate(groups):
            in_g = np.isin(yr, g)
            for b in range(N_BLOCKS):
                test = in_g & (blk == b)
                train = (~in_g) & (~buffered_exclude(b))
                folds.append((f"yg{gi}_b{b}", train, test))
    else:
        raise ValueError(scheme)
    return [f for f in folds if f[2].sum() > 0 and f[1].sum() > 0]


# ------------------------- models -------------------------
def make_model(name, pos_weight):
    if name == "xgb":
        try:
            from xgboost import XGBClassifier
            return XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8,
                                 colsample_bytree=0.8, tree_method="hist", scale_pos_weight=pos_weight,
                                 eval_metric="logloss", n_jobs=-1, random_state=SEED), False
        except ImportError:
            print("xgboost not installed -> using HistGradientBoosting instead")
            name = "hgb"
    if name == "hgb":
        return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.06, max_leaf_nodes=31,
                                              class_weight="balanced", random_state=SEED), False
    if name == "rf":
        return RandomForestClassifier(n_estimators=200, min_samples_leaf=30, max_features="sqrt",
                                      class_weight="balanced_subsample", n_jobs=-1, random_state=SEED), True
    raise ValueError(name)


# ------------------------- metrics -------------------------
def score_metrics(y, s):
    ok = ~np.isnan(s)
    y, s = y[ok], s[ok]
    if len(np.unique(y)) < 2:
        return dict(pr_auc=np.nan, lift=np.nan, roc_auc=np.nan, precision_r70=np.nan, far_r70=np.nan, n=len(y))
    base = y.mean()
    pr = average_precision_score(y, s)
    prec, rec, _ = precision_recall_curve(y, s)
    p70 = prec[rec >= 0.7].max() if (rec >= 0.7).any() else np.nan
    return dict(pr_auc=pr, lift=pr / base, roc_auc=roc_auc_score(y, s),
                precision_r70=p70, far_r70=1 - p70, n=len(y), base_rate=base)


# ------------------------- main loop -------------------------
MIN_CLASS = 30   # min positives AND negatives in a fold for a metric to count


def fold_metric_rows(H, subset, mask, scores, y_all, folds):
    rows = []
    for fname, _, te in folds:
        m = te & mask
        yt = y_all[m]
        if yt.sum() < MIN_CLASS or (len(yt) - yt.sum()) < MIN_CLASS:
            continue
        for (sname, mname), sc in scores.items():
            r = score_metrics(yt, sc[m])
            r.update(dict(horizon_days=16 * H, subset=subset, set=sname, model=mname, fold=fname))
            rows.append(r)
    return rows


def pooled_rows(H, subset, mask, scores, y_all):
    rows = []
    for (sname, mname), sc in scores.items():
        r = score_metrics(y_all[mask], sc[mask])
        r.update(dict(horizon_days=16 * H, subset=subset, set=sname, model=mname))
        rows.append(r)
    return rows


def summarise_folds(fm):
    keys = ["horizon_days", "subset", "fold"]
    pers = fm[(fm["set"] == "baseline") & (fm["model"] == "persistence")][keys + ["lift", "roc_auc"]]
    pers = pers.rename(columns={"lift": "lift_pers", "roc_auc": "roc_pers"})
    d = fm.merge(pers, on=keys, how="left")
    d["d_lift"] = d["lift"] - d["lift_pers"]
    d["d_roc"] = d["roc_auc"] - d["roc_pers"]
    g = d.groupby(["horizon_days", "subset", "set", "model"])
    out = g.agg(n_folds=("fold", "count"),
                lift_mean=("lift", "mean"), lift_sd=("lift", "std"),
                roc_mean=("roc_auc", "mean"), roc_sd=("roc_auc", "std"),
                far_r70_mean=("far_r70", "mean"),
                d_lift_vs_pers=("d_lift", "mean"), d_roc_vs_pers=("d_roc", "mean"),
                folds_beating_pers=("d_lift", lambda s: float((s > 0).mean()))).reset_index()
    return out


def run(feat_dir, label_dir, out_dir, horizons, sets, models, cv, max_train, n_year_groups,
        feat_file="features_long", min_year=None, eval_min_year=None, target="stress"):
    os.makedirs(out_dir, exist_ok=True)
    rng = np.random.default_rng(SEED)
    feat, lab, cells = prepare(feat_dir, label_dir, feat_file, min_year, target)
    y_all = lab["y"].to_numpy()
    folds = make_folds(lab, cells, cv, n_year_groups=n_year_groups, eval_min_year=eval_min_year)
    eval_ok = (lab["year"] >= eval_min_year).to_numpy() if eval_min_year else np.ones(len(lab), dtype=bool)
    print(f"CV scheme '{cv}': {len(folds)} folds")

    fold_rows, pooled = [], []
    for H in horizons:
        t0 = time.time()
        Xc, Xn = build_xy(feat, lab, H)
        X_sets = {"climate": Xc, "ndvi": pd.concat([Xn, Xc[["k"]]], axis=1),
                  "climate+ndvi": pd.concat([Xc, Xn], axis=1)}
        if any("s1" in sn for sn in sets):
            Xs = build_s1(feat, lab, H)
            kcol = Xc[["k"]]
            X_sets.update({"s1": pd.concat([Xs, kcol], axis=1),
                           "ndvi+s1": pd.concat([Xn, Xs, kcol], axis=1),
                           "climate+s1": pd.concat([Xc, Xs], axis=1),
                           "climate+ndvi+s1": pd.concat([Xc, Xn, Xs], axis=1)})

        # evaluation subsets (training always uses all rows)
        z_issue = Xn[f"ndvi_z_L{H + 1}"].to_numpy(np.float64)   # last known NDVI z at issue time
        pers = -z_issue
        subsets = {"all": ~np.isnan(pers) & eval_ok,
                   "onset": ~np.isnan(pers) & (z_issue >= -1.0) & eval_ok}   # not yet stressed at issue time
        print(f"\n=== H = {H} windows ({16 * H} days) | eval rows all {subsets['all'].sum():,}, "
              f"onset {subsets['onset'].sum():,} (onset base rate "
              f"{y_all[subsets['onset']].mean():.3f}) ===")

        scores = {("baseline", "persistence"): pers,
                  ("baseline", "spi_rule"): -Xc[f"spi32_L{H}"].to_numpy(np.float64)}
        clim_oof = np.full(len(lab), np.nan)
        for _, tr, te in folds:
            rate = lab.loc[tr].groupby("k")["y"].mean()
            clim_oof[te] = lab.loc[te, "k"].map(rate).to_numpy()
        scores[("baseline", "climatology")] = clim_oof

        for sname in sets:
            X = X_sets[sname]
            for mname in models:
                oof = np.full(len(lab), np.nan)
                for fname, tr, te in folds:
                    tr_idx = np.flatnonzero(tr)
                    if len(tr_idx) > max_train:
                        tr_idx = rng.choice(tr_idx, max_train, replace=False)
                    ytr = y_all[tr_idx]
                    if ytr.sum() < 20:
                        continue
                    model, needs_fill = make_model(mname, (len(ytr) - ytr.sum()) / max(ytr.sum(), 1))
                    Xtr, Xte = X.iloc[tr_idx], X[te]
                    if needs_fill:
                        Xtr, Xte = Xtr.fillna(-999), Xte.fillna(-999)
                    model.fit(Xtr, ytr)
                    oof[te] = model.predict_proba(Xte)[:, 1]
                scores[(sname, mname)] = oof
                print(f"  trained {sname} / {mname}")

        for sub, mask in subsets.items():
            fold_rows += fold_metric_rows(H, sub, mask, scores, y_all, folds)
            pooled += pooled_rows(H, sub, mask, scores, y_all)

        oof_df = lab[["pid", "year", "k"]].copy()
        oof_df["y"] = y_all
        oof_df["onset"] = subsets["onset"]
        oof_df["eval_all"] = subsets["all"]
        for (sname, mname), sc in scores.items():
            if sname != "baseline":
                oof_df[f"{sname}|{mname}"] = sc.astype(np.float32)
        oof_df.to_csv(os.path.join(out_dir, f"oof_H{16 * H}d.csv.gz"), index=False)
        print(f"  H={H} took {(time.time() - t0) / 60:.1f} min")

    fm = pd.DataFrame(fold_rows)
    fm.to_csv(os.path.join(out_dir, "fold_metrics.csv"), index=False)
    if fm.empty:
        print("\nNo fold had at least %d positive and %d negative test rows, so no metrics were computed. "
              "(Expected only for very rare targets or tiny data.)" % (MIN_CLASS, MIN_CLASS))
        return None
    pd.DataFrame(pooled).to_csv(os.path.join(out_dir, "pooled_metrics.csv"), index=False)
    summ = summarise_folds(fm)
    summ.to_csv(os.path.join(out_dir, "summary_by_horizon.csv"), index=False)
    plot(summ, os.path.join(out_dir, "skill_vs_leadtime.png"))

    for sub in ["all", "onset"]:
        print(f"\n##### Fold-wise summary, subset = {sub} #####")
        cols = ["horizon_days", "set", "model", "n_folds", "lift_mean", "lift_sd", "roc_mean",
                "d_lift_vs_pers", "folds_beating_pers"]
        print(summ[summ["subset"] == sub][cols].round(3).to_string(index=False))
    return summ


def plot(summ, path):
    fig, ax = plt.subplots(2, 2, figsize=(13, 8))
    for j, sub in enumerate(["all", "onset"]):
        d = summ[summ["subset"] == sub]
        for (s, m), g in d.groupby(["set", "model"]):
            g = g.sort_values("horizon_days")
            ls = "--" if s == "baseline" else "-"
            ax[0, j].errorbar(g["horizon_days"], g["lift_mean"], yerr=g["lift_sd"], fmt=ls, marker="o",
                              ms=4, capsize=2, label=f"{s}/{m}")
            ax[1, j].errorbar(g["horizon_days"], g["roc_mean"], yerr=g["roc_sd"], fmt=ls, marker="o",
                              ms=4, capsize=2, label=f"{s}/{m}")
        ax[0, j].axhline(1, color="k", lw=0.6)
        ax[1, j].axhline(0.5, color="k", lw=0.6)
        title = "all rows" if sub == "all" else "onset only (not stressed at issue time)"
        ax[0, j].set_title(f"PR-AUC lift, mean +- sd over folds: {title}", fontsize=10)
        ax[1, j].set_title(f"ROC-AUC: {title}", fontsize=10)
        ax[1, j].set_xlabel("Lead time (days)")
    ax[0, 0].legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--feat-dir", required=True)
    ap.add_argument("--label-dir", required=True)
    ap.add_argument("--out-dir", required=True)
    ap.add_argument("--horizons", nargs="*", type=int, default=[0, 1, 2, 3], help="in 16-day windows")
    ap.add_argument("--sets", nargs="*", default=["climate", "climate+ndvi"],
                    choices=["climate", "ndvi", "climate+ndvi", "s1", "ndvi+s1", "climate+s1", "climate+ndvi+s1"])
    ap.add_argument("--models", nargs="*", default=["xgb"], choices=["xgb", "rf", "hgb"])
    ap.add_argument("--cv", default="st", choices=["st", "loso", "spatial"])
    ap.add_argument("--max-train", type=int, default=200_000)
    ap.add_argument("--year-groups", type=int, default=N_YEAR_GROUPS)
    ap.add_argument("--feat-file", default="features_long", help="features_long_s1 for the Sentinel-1 sets")
    ap.add_argument("--min-year", type=int, default=None, help="use 2016 with Sentinel-1 so every set uses the same rows")
    ap.add_argument("--target", default="stress", choices=["stress", "sustained"],
                    help="stress: z < -1 in the window; sustained: z < -1 in this window AND the next")
    ap.add_argument("--eval-min-year", type=int, default=None,
                    help="only test on years >= this but TRAIN on all years (preferred over --min-year for the S1 test)")
    ap.add_argument("--quick", action="store_true", help="3 year groups, 100k training rows, horizons 0-2")
    a = ap.parse_args(argv)
    if a.quick:
        a.max_train, a.year_groups, a.horizons = 100_000, 3, [0, 1, 2]
    return run(a.feat_dir, a.label_dir, a.out_dir, a.horizons, a.sets, a.models, a.cv, a.max_train, a.year_groups,
               a.feat_file, a.min_year, a.eval_min_year, a.target)


if __name__ == "__main__" and "ipykernel" not in sys.modules:
    main()

In [4]:
main(["--feat-dir", r"crop_stress\features", "--feat-file", "features_long_s1",
      "--label-dir", r"crop_stress\label", "--out-dir", r"crop_stress\models_sust",
      "--sets", "ndvi", "climate+ndvi", "ndvi+s1", "climate+ndvi+s1",
      "--models", "xgb", "--eval-min-year", "2016", "--year-groups", "5",
      "--horizons", "0", "1", "2", "--target", "sustained"])

Target = SUSTAINED stress (z < -1 now and in the next 16-day window)


C:\Users\DELL\anaconda3\envs\geospatial\Lib\site-packages\sklearn\cluster\_kmeans.py:1424: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=3.
  warnings.warn(


Rows: 470,853  points: 3,185  cells: 572  base stress rate: 0.105
CV scheme 'st': 25 folds

=== H = 0 windows (0 days) | eval rows all 155,528, onset 106,655 (onset base rate 0.083) ===
  trained ndvi / xgb
  trained climate+ndvi / xgb
  trained ndvi+s1 / xgb
  trained climate+ndvi+s1 / xgb
  H=0 took 13.9 min

=== H = 1 windows (16 days) | eval rows all 121,671, onset 82,669 (onset base rate 0.098) ===
  trained ndvi / xgb
  trained climate+ndvi / xgb
  trained ndvi+s1 / xgb
  trained climate+ndvi+s1 / xgb
  H=1 took 13.2 min

=== H = 2 windows (32 days) | eval rows all 97,407, onset 65,362 (onset base rate 0.103) ===
  trained ndvi / xgb
  trained climate+ndvi / xgb
  trained ndvi+s1 / xgb
  trained climate+ndvi+s1 / xgb
  H=2 took 12.9 min

##### Fold-wise summary, subset = all #####
 horizon_days             set       model  n_folds  lift_mean  lift_sd  roc_mean  d_lift_vs_pers  folds_beating_pers
            0        baseline climatology       25      1.053    0.222     0.497     

,horizon_days,subset,set,model,n_folds,lift_mean,lift_sd,roc_mean,roc_sd,far_r70_mean,d_lift_vs_pers,d_roc_vs_pers,folds_beating_pers
0,0,all,baseline,climatology,25,1.053281,0.221977,0.497097,0.086180,0.844513,-1.140225,-0.257793,0.04
1,0,all,baseline,persistence,25,2.193506,0.416368,0.754889,0.050275,0.731291,0.000000,0.000000,0.00
2,0,all,baseline,spi_rule,25,1.302926,0.423479,0.568612,0.106250,0.813471,-0.890580,-0.186278,0.12
3,0,all,climate+ndvi,xgb,25,2.657313,0.854726,0.774927,0.051804,0.717078,0.463807,0.020038,0.84
4,0,all,climate+ndvi+s1,xgb,25,2.788516,0.810244,0.784441,0.046523,0.714975,0.595010,0.029552,0.88
5,0,all,ndvi,xgb,25,2.392798,0.701391,0.756522,0.054708,0.730765,0.199293,0.001633,0.64
6,0,all,ndvi+s1,xgb,25,2.507277,0.750530,0.764879,0.049730,0.727169,0.313771,0.009990,0.76
7,0,onset,baseline,climatology,25,1.128660,0.492172,0.470360,0.119421,0.913732,-1.155897,-0.261091,0.04
8,0,onset,baseline,persistence,25,2.284557,0.590846,0.731452,0.053774,0.851087,0.000000,0.000000,0.00
9,0,onset,baseline,spi_rule,25,1.595334,0.747377,0.605980,0.096972,0.883582,-0.689223,-0.125471,0.12
